# Condition 2 Pilot Analysis (2026-03-13)

Second condition 2 pilot with two design fixes applied:
1. **Double study pass**: all 60 study pairs presented twice (120 study trials total), to strengthen associative memory traces
2. **30/30 split fix**: intact/rearranged allocation bug corrected, now 30 intact + 30 rearranged (10/10 per emotion)

Data: 7 subjects total (6 condition 2, 1 condition 1). Analysis restricted to condition 2.

**Comparison to Mar 11 pilot (n=6):** The Mar 11 data used the old design (single study pass, 24/36 split bug) and showed zero associative discrimination (d' near zero). The key question: does doubling the study exposure improve discrimination?

### Analyses

**Study phase (orienting).** 2(flanker gender compatibility) x 3(flanker emotion) RM ANOVAs on accuracy and RT.

**Test phase (associative recognition).** 2(pair type: intact/rearranged) x 3(flanker emotion) RM ANOVAs on p("same") and RT.

**Supplementary.** Signal detection analysis (d', criterion) per emotion.

In [1]:
import pandas as pd
from pathlib import Path
from statistics import NormalDist

z = NormalDist().inv_cdf

_nb_dir = Path.cwd() if '__vsc_ipynb_file__' not in dir() else Path(__vsc_ipynb_file__).parent
df = pd.read_csv(_nb_dir / '2026_03_13_pilot_7subj_c2.csv')
df['correct'] = df['correct'].astype('boolean')

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')
print()

# Filter to condition 2 only
df = df[df.condition == 2].copy()
print(f'After filtering to condition 2: {len(df)} rows, {df.subject_number.nunique()} subjects')
print()

# Verify study-phase double presentation
study_counts = df[df.phase == 'study'].groupby('subject_number').size()
print(f'Study trials per subject: {study_counts.unique()} (expected 120 = 60 pairs x 2 passes)')
print()

# Verify test-phase split
test_all = df[df.phase == 'test']
pt_counts = test_all.groupby(['subject_number', 'pair_type']).size().unstack(fill_value=0)
print('Test-phase intact/rearranged per subject:')
print(pt_counts.to_string())
print()
emo_pt = test_all.groupby(['flanker_emotion', 'pair_type']).size().unstack(fill_value=0)
print('Test-phase counts by emotion x pair_type (all subjects):')
print(emo_pt.to_string())

1260 rows, 7 subjects
Conditions: {1: 1, 2: 6}

After filtering to condition 2: 1080 rows, 6 subjects

Study trials per subject: [120] (expected 120 = 60 pairs x 2 passes)

Test-phase intact/rearranged per subject:
pair_type       intact  rearranged
subject_number                    
1                   30          30
2                   30          30
3                   30          30
4                   30          30
5                   30          30
6                   30          30

Test-phase counts by emotion x pair_type (all subjects):
pair_type        intact  rearranged
flanker_emotion                    
angry                60          60
happy                60          60
neutral              60          60


## Exclusion Criteria

Same criterion as condition 1: exclude subjects with >=6 zero-correct cells out of 12 in the study-phase design (2 target gender x 2 flanker gender x 3 flanker emotion). With the double study pass, each cell now has 10 trials (vs 5 in the Mar 11 pilot).

In [2]:
ZERO_CELL_THRESHOLD = 6

study_all = df[df.phase == 'study']

cell_correct = study_all.groupby(
    ['subject_number', 'target_gender', 'flanker_gender', 'flanker_emotion']
).correct.sum().reset_index()
zero_cells = cell_correct.groupby('subject_number').apply(
    lambda g: (g.correct == 0).sum()
).reset_index(name='zero_cells')

subj_study_acc = study_all.groupby('subject_number').correct.mean()
zero_cells['study_accuracy'] = zero_cells.subject_number.map(subj_study_acc)

trials_per_cell = study_all.groupby(
    ['subject_number', 'target_gender', 'flanker_gender', 'flanker_emotion']
).size().iloc[0]
print(f'Study-phase design: 12 cells per subject ({trials_per_cell} trials each)')
print(f'Subjects with zero-correct cells:')
has_zeros = zero_cells[zero_cells.zero_cells > 0].sort_values('zero_cells', ascending=False)
if len(has_zeros) == 0:
    print('  None')
else:
    for _, row in has_zeros.iterrows():
        flag = ' ** EXCLUDED' if row.zero_cells >= ZERO_CELL_THRESHOLD else ''
        print(f'  Subject {int(row.subject_number)}: '
              f'{int(row.zero_cells)}/12 zero cells, {row.study_accuracy:.1%} accuracy{flag}')
print()

excluded = zero_cells[zero_cells.zero_cells >= ZERO_CELL_THRESHOLD].subject_number.tolist()
keep = zero_cells[zero_cells.zero_cells < ZERO_CELL_THRESHOLD].subject_number.tolist()
df = df[df.subject_number.isin(keep)].copy()
print(f'{len(excluded)} excluded, {df.subject_number.nunique()} subjects remain')

Study-phase design: 12 cells per subject (10 trials each)
Subjects with zero-correct cells:
  Subject 1: 4/12 zero cells, 48.3% accuracy
  Subject 5: 4/12 zero cells, 50.8% accuracy

0 excluded, 6 subjects remain


### Exclusion summary

No subjects excluded. Two subjects (1 and 5) had 4/12 zero-correct cells with study accuracy near 50%, suggesting chance-level responding on incompatible trials. However, neither met the 6-cell threshold.

With 10 trials per cell (double the Mar 11 pilot's 5), zero-correct cells are less likely for engaged participants, so the criterion is effectively more lenient here. All 6 condition 2 subjects are retained for analysis.

## Study Phase (Orienting)

2(flanker gender compatibility) x 3(flanker emotion) RM ANOVAs on accuracy and RT. Each cell has 10 trials (double the Mar 11 pilot due to the second study pass).

In [3]:
study = df[df.phase == 'study'].copy()
study['compatible'] = study.target_gender == study.flanker_gender
study['compat_label'] = study.compatible.map({True: 'compatible', False: 'incompatible'})

study_acc_subj = study.groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).correct.mean().reset_index(name='accuracy')

study_rt_subj = study[~study.timed_out].groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

acc_table = study_acc_subj.groupby(['compat_label', 'flanker_emotion']).accuracy.agg(
    ['mean', 'std']
).round(3)
print('Study-phase accuracy by compatibility x emotion:')
print(acc_table.to_string())
print()

print('Marginal means (accuracy):')
print(f'  Compatible:   {study_acc_subj[study_acc_subj.compat_label == "compatible"].accuracy.mean():.3f}')
print(f'  Incompatible: {study_acc_subj[study_acc_subj.compat_label == "incompatible"].accuracy.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_acc_subj[study_acc_subj.flanker_emotion == emo].accuracy.mean():.3f}')
print()

rt_table = study_rt_subj.groupby(['compat_label', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print('Study-phase RT (ms) by compatibility x emotion:')
print(rt_table.to_string())
print()

print('Marginal means (RT):')
print(f'  Compatible:   {study_rt_subj[study_rt_subj.compat_label == "compatible"].mean_rt.mean():.1f}')
print(f'  Incompatible: {study_rt_subj[study_rt_subj.compat_label == "incompatible"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_rt_subj[study_rt_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Study-phase accuracy by compatibility x emotion:
                               mean    std
compat_label flanker_emotion              
compatible   angry             0.95  0.045
             happy            0.933  0.061
             neutral           0.95  0.055
incompatible angry            0.592  0.388
             happy            0.575  0.459
             neutral           0.55  0.443

Marginal means (accuracy):
  Compatible:   0.944
  Incompatible: 0.572
         angry: 0.771
         happy: 0.754
       neutral: 0.750

Study-phase RT (ms) by compatibility x emotion:
                                mean    std
compat_label flanker_emotion               
compatible   angry             980.9  191.1
             happy            1005.5  197.8
             neutral           973.8  192.1
incompatible angry            1299.1  563.4
             happy            1172.2  334.2
             neutral          1272.4  395.2

Marginal means (RT):
  Compatible:   986.7
  Incompatible: 1247.9
 

In [4]:
import math

def _betacf(a, b, x):
    MAXIT, EPS = 200, 3e-12
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 / (1.0 - qab * x / qap) if abs(1.0 - qab * x / qap) > 1e-30 else 1.0 / 1e-30
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < EPS:
            break
    return h

def _betai(a, b, x):
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a * math.log(x) + b * math.log(1 - x) - lbeta)
    if x < (a + 1) / (a + b + 2):
        return front * _betacf(a, b, x) / a
    else:
        return 1.0 - front * _betacf(b, a, 1 - x) / b

def t_p_twotail(t_val, df):
    x = df / (df + t_val ** 2)
    return _betai(df / 2.0, 0.5, x)

def f_p(f_val, df1, df2):
    if f_val <= 0: return 1.0
    x = df2 / (df2 + df1 * f_val)
    return _betai(df2 / 2.0, df1 / 2.0, x)

def rm_anova_oneway(groups):
    k = len(groups)
    n = len(groups[0])
    grand = sum(sum(g) for g in groups) / (k * n)
    subj_m = [sum(groups[j][i] for j in range(k)) / k for i in range(n)]
    cond_m = [sum(g) / n for g in groups]
    ss_cond = n * sum((m - grand) ** 2 for m in cond_m)
    ss_subj = k * sum((m - grand) ** 2 for m in subj_m)
    ss_total = sum((groups[j][i] - grand) ** 2 for j in range(k) for i in range(n))
    ss_err = ss_total - ss_cond - ss_subj
    df1 = k - 1
    df2 = (k - 1) * (n - 1)
    ms_err = ss_err / df2 if df2 > 0 else float('nan')
    f_val = (ss_cond / df1) / ms_err if ss_err > 0 else float('nan')
    p = f_p(f_val, df1, df2)
    eta = ss_cond / (ss_cond + ss_err)
    return f_val, df1, df2, p, eta, ms_err

def rm_anova_twoway(data, a_levels, b_levels):
    a = len(a_levels)
    b = len(b_levels)
    n = len(data[(a_levels[0], b_levels[0])])
    Y = [[[data[(a_levels[j], b_levels[k])][i]
           for k in range(b)] for j in range(a)] for i in range(n)]
    gm = sum(Y[i][j][k] for i in range(n) for j in range(a) for k in range(b)) / (n * a * b)
    subj_m = [sum(Y[i][j][k] for j in range(a) for k in range(b)) / (a * b) for i in range(n)]
    a_m = [sum(Y[i][j][k] for i in range(n) for k in range(b)) / (n * b) for j in range(a)]
    b_m = [sum(Y[i][j][k] for i in range(n) for j in range(a)) / (n * a) for k in range(b)]
    ab_m = [[sum(Y[i][j][k] for i in range(n)) / n for k in range(b)] for j in range(a)]
    sa_m = [[sum(Y[i][j][k] for k in range(b)) / b for j in range(a)] for i in range(n)]
    sb_m = [[sum(Y[i][j][k] for j in range(a)) / a for k in range(b)] for i in range(n)]
    ss_a = n * b * sum((a_m[j] - gm) ** 2 for j in range(a))
    ss_b = n * a * sum((b_m[k] - gm) ** 2 for k in range(b))
    ss_ab = n * sum((ab_m[j][k] - a_m[j] - b_m[k] + gm) ** 2
                    for j in range(a) for k in range(b))
    ss_s = a * b * sum((subj_m[i] - gm) ** 2 for i in range(n))
    ss_as = b * sum((sa_m[i][j] - a_m[j] - subj_m[i] + gm) ** 2
                    for i in range(n) for j in range(a))
    ss_bs = a * sum((sb_m[i][k] - b_m[k] - subj_m[i] + gm) ** 2
                    for i in range(n) for k in range(b))
    ss_total = sum((Y[i][j][k] - gm) ** 2
                   for i in range(n) for j in range(a) for k in range(b))
    ss_abs = ss_total - ss_a - ss_b - ss_ab - ss_s - ss_as - ss_bs
    df_a, df_b, df_ab = a - 1, b - 1, (a - 1) * (b - 1)
    df_as, df_bs, df_abs = df_a * (n - 1), df_b * (n - 1), df_ab * (n - 1)
    results = {}
    for label, ss_eff, df_eff, ss_e, df_e in [
        ('A', ss_a, df_a, ss_as, df_as),
        ('B', ss_b, df_b, ss_bs, df_bs),
        ('AxB', ss_ab, df_ab, ss_abs, df_abs),
    ]:
        ms_eff = ss_eff / df_eff if df_eff > 0 else 0
        ms_e = ss_e / df_e if df_e > 0 else float('nan')
        f_val = ms_eff / ms_e if ms_e > 0 else float('nan')
        p = f_p(f_val, df_eff, df_e)
        eta = ss_eff / (ss_eff + ss_e) if (ss_eff + ss_e) > 0 else 0
        results[label] = {
            'F': f_val, 'df1': df_eff, 'df2': df_e,
            'p': p, 'eta_sq': eta, 'ms_error': ms_e
        }
    return results

def anova_followup(means_a, means_b, ms_error, df_error, label_a, label_b):
    n = len(means_a)
    diff = sum(a - b for a, b in zip(means_a, means_b)) / n
    se = math.sqrt(2 * ms_error / n)
    if se == 0:
        return 0.0, df_error, 1.0, diff
    t_val = diff / se
    p = t_p_twotail(t_val, df_error)
    return t_val, df_error, p, diff

def edge_correct(rate, n):
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate


# --- Study-phase 2x3 ANOVAs ---

subjects = sorted(study_acc_subj.subject_number.unique())
n_subj = len(subjects)
a_levels = ['compatible', 'incompatible']
b_levels = ['angry', 'happy', 'neutral']

acc_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_acc_subj.compat_label == al) & (study_acc_subj.flanker_emotion == bl)
        vals = study_acc_subj[mask].set_index('subject_number').loc[subjects, 'accuracy'].tolist()
        acc_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on accuracy (n={n_subj}):')
print()
acc_results = rm_anova_twoway(acc_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = acc_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

rt_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_rt_subj.compat_label == al) & (study_rt_subj.flanker_emotion == bl)
        vals = study_rt_subj[mask].set_index('subject_number').loc[subjects, 'mean_rt'].tolist()
        rt_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on RT (n={n_subj}):')
print()
rt_results = rm_anova_twoway(rt_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = rt_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")

2(compatibility) x 3(emotion) RM ANOVA on accuracy (n=6):

  Compatibility: F(1,5) = 4.730, p = 0.082, partial eta^2 = 0.486
  Emotion: F(2,10) = 0.354, p = 0.711, partial eta^2 = 0.066
  Compatibility x Emotion: F(2,10) = 0.723, p = 0.509, partial eta^2 = 0.126

2(compatibility) x 3(emotion) RM ANOVA on RT (n=6):

  Compatibility: F(1,5) = 2.725, p = 0.160, partial eta^2 = 0.353
  Emotion: F(2,10) = 0.381, p = 0.693, partial eta^2 = 0.071
  Compatibility x Emotion: F(2,10) = 2.079, p = 0.176, partial eta^2 = 0.294


### Study phase interpretation

The flanker-compatibility pattern replicates the Mar 11 pilot and condition 1 results. Compatible trials show high accuracy (.94) while incompatible trials are much lower (.57), with large variability (SDs .39-.46) reflecting individual differences in whether subjects used the flanker face or ignored it. The compatibility effect approached significance, F(1,5) = 4.73, p = .082, with a large effect size (eta^2 = .49). No main effect of emotion and no interaction (both Fs < 1).

RT shows the same directional pattern: compatible faster (987 ms) than incompatible (1248 ms), though not significant, F(1,5) = 2.73, p = .160. The large variability in incompatible RTs (SDs 334-563) again reflects individual strategy differences.

These study-phase results confirm that the orienting task is working as intended and that the double study pass did not change how subjects approached the encoding task.

## Test Phase (Associative Recognition)

Each test trial shows a face pair. Intact pairs are the same target-flanker combination seen at study; rearranged pairs swap the flanker identity within the same trial type. The participant judges "same" (intact) or "different" (rearranged).

2(pair type: intact/rearranged) x 3(flanker emotion) RM ANOVAs on p("same") and RT.

**Cell sizes.** 10 intact + 10 rearranged per emotion per subject (balanced, unlike the Mar 11 pilot's 8/12).

In [5]:
test = df[df.phase == 'test'].copy()

test['said_same'] = test.apply(
    lambda r: bool(r.correct) if r.pair_type == 'intact' else not bool(r.correct), axis=1
)

psame_subj = test.groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).said_same.mean().reset_index(name='p_same')

rt_test_subj = test[~test.timed_out].groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

print('Test-phase p("same") by pair_type x emotion:')
psame_table = psame_subj.groupby(['pair_type', 'flanker_emotion']).p_same.agg(
    ['mean', 'std']
).round(3)
print(psame_table.to_string())
print()

print('Marginal means p("same"):')
print(f'  Intact:     {psame_subj[psame_subj.pair_type == "intact"].p_same.mean():.3f}')
print(f'  Rearranged: {psame_subj[psame_subj.pair_type == "rearranged"].p_same.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {psame_subj[psame_subj.flanker_emotion == emo].p_same.mean():.3f}')
print()

print('Test-phase RT (ms) by pair_type x emotion:')
rt_table = rt_test_subj.groupby(['pair_type', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print(rt_table.to_string())
print()

print('Marginal means RT (ms):')
print(f'  Intact:     {rt_test_subj[rt_test_subj.pair_type == "intact"].mean_rt.mean():.1f}')
print(f'  Rearranged: {rt_test_subj[rt_test_subj.pair_type == "rearranged"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {rt_test_subj[rt_test_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Test-phase p("same") by pair_type x emotion:
                             mean    std
pair_type  flanker_emotion              
intact     angry            0.517  0.204
           happy            0.483  0.183
           neutral          0.517  0.117
rearranged angry            0.533  0.225
           happy            0.517  0.147
           neutral          0.467  0.175

Marginal means p("same"):
  Intact:     0.506
  Rearranged: 0.506
         angry: 0.525
         happy: 0.500
       neutral: 0.492

Test-phase RT (ms) by pair_type x emotion:
                              mean    std
pair_type  flanker_emotion               
intact     angry             993.4  352.5
           happy             937.3  394.2
           neutral          1004.0  421.0
rearranged angry             970.6  262.0
           happy             928.2  323.8
           neutral           994.7  352.2

Marginal means RT (ms):
  Intact:     978.2
  Rearranged: 964.5
         angry: 982.0
         happy: 932.7
     

In [6]:
test_subjects = sorted(psame_subj.subject_number.unique())
n_test = len(test_subjects)
pt_levels = ['intact', 'rearranged']
emo_levels = ['angry', 'happy', 'neutral']

psame_data = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (psame_subj.pair_type == pt) & (psame_subj.flanker_emotion == emo)
        vals = psame_subj[mask].set_index('subject_number').loc[test_subjects, 'p_same'].tolist()
        psame_data[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on p("same") (n={n_test}):')
print()
psame_results = rm_anova_twoway(psame_data, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = psame_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

rt_data_test = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (rt_test_subj.pair_type == pt) & (rt_test_subj.flanker_emotion == emo)
        vals = rt_test_subj[mask].set_index('subject_number').loc[test_subjects, 'mean_rt'].tolist()
        rt_data_test[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on RT (n={n_test}):')
print()
rt_test_results = rm_anova_twoway(rt_data_test, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = rt_test_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

mse_int_psame = psame_results['AxB']['ms_error']
df_int_psame = psame_results['AxB']['df2']
mse_int_rt = rt_test_results['AxB']['ms_error']
df_int_rt = rt_test_results['AxB']['df2']

print('Follow-up comparisons on p("same"):')
print()
print('  Intact vs rearranged within each emotion (discrimination):')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', emo)], psame_data[('rearranged', emo)],
        mse_int_psame, df_int_psame, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within intact (hit rate modulation):')
emo_pairs = [('angry', 'happy'), ('angry', 'neutral'), ('happy', 'neutral')]
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', e1)], psame_data[('intact', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within rearranged (FA rate modulation):')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('rearranged', e1)], psame_data[('rearranged', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('Follow-up comparisons on RT:')
print()
print('  Intact vs rearranged within each emotion:')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', emo)], rt_data_test[('rearranged', emo)],
        mse_int_rt, df_int_rt, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within intact:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', e1)], rt_data_test[('intact', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within rearranged:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('rearranged', e1)], rt_data_test[('rearranged', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')

2(pair_type) x 3(emotion) RM ANOVA on p("same") (n=6):

  Pair type: F(1,5) = 0.000, p = 1.000, partial eta^2 = 0.000
  Emotion: F(2,10) = 0.175, p = 0.842, partial eta^2 = 0.034
  Pair type x Emotion: F(2,10) = 0.432, p = 0.661, partial eta^2 = 0.080

2(pair_type) x 3(emotion) RM ANOVA on RT (n=6):

  Pair type: F(1,5) = 0.108, p = 0.755, partial eta^2 = 0.021
  Emotion: F(2,10) = 0.732, p = 0.505, partial eta^2 = 0.128
  Pair type x Emotion: F(2,10) = 0.024, p = 0.976, partial eta^2 = 0.005

Follow-up comparisons on p("same"):

  Intact vs rearranged within each emotion (discrimination):
    angry: t(10) = -0.248, p = 0.809, diff = -0.017
    happy: t(10) = -0.497, p = 0.630, diff = -0.033
    neutral: t(10) = 0.745, p = 0.473, diff = 0.050

  Pairwise emotion within intact (hit rate modulation):
    angry vs happy: t(10) = 0.497, p = 0.630, diff = 0.033
    angry vs neutral: t(10) = 0.000, p = 1.000, diff = 0.000
    happy vs neutral: t(10) = -0.497, p = 0.630, diff = -0.033

  Pair

### Test phase interpretation

**Discrimination remains at zero despite double study exposure.** The pair type main effect on p("same") is F(1,5) = 0.000, p = 1.000 -- subjects responded "same" to exactly 50.6% of both intact and rearranged pairs. There is no hint of discrimination in any emotion condition (all follow-up ts < 1). No emotion effects and no interaction (all Fs < 1).

RT shows the same null pattern: no pair type effect (F < 1), no emotion effect (F < 1), no interaction (F < 1). Intact (978 ms) and rearranged (965 ms) RTs are virtually identical.

**Comparison to Mar 11 pilot.** The Mar 11 data (single study pass, 24/36 bug) also showed zero discrimination. The current data, collected after both fixes (double study pass giving 120 study trials, balanced 30/30 test split), shows no improvement whatsoever. Doubling the number of study exposures did not produce any measurable associative memory signal.

## Supplementary: Signal Detection Analysis

d' and criterion per emotion. Hit = p("same" | intact), FA = p("same" | rearranged). Edge correction: Macmillan & Kaplan (1985). Cell sizes: 10 intact + 10 rearranged per emotion per subject.

In [7]:
sdt_per_subj = []
for subj in test_subjects:
    sdata = test[test.subject_number == subj]
    for emotion in emo_levels:
        intact_emo = sdata[(sdata.pair_type == 'intact') & (sdata.flanker_emotion == emotion)]
        rearr_emo = sdata[(sdata.pair_type == 'rearranged') & (sdata.flanker_emotion == emotion)]

        n_intact = len(intact_emo)
        n_rearr = len(rearr_emo)

        hit_rate_raw = intact_emo.said_same.mean() if n_intact > 0 else 0.0
        fa_rate_raw = rearr_emo.said_same.mean() if n_rearr > 0 else 0.0

        hit_rate = edge_correct(hit_rate_raw, n_intact) if n_intact > 0 else 0.5
        fa_rate = edge_correct(fa_rate_raw, n_rearr) if n_rearr > 0 else 0.5

        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))

        sdt_per_subj.append({
            'subject': subj,
            'emotion': emotion,
            'n_intact': n_intact,
            'n_rearranged': n_rearr,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

sdt_df = pd.DataFrame(sdt_per_subj)

sdt_summary = sdt_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    hit_rate_SD=('hit_rate', 'std'),
    fa_rate_M=('fa_rate', 'mean'),
    fa_rate_SD=('fa_rate', 'std'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'SDT Analysis (n={n_test} subjects, per-emotion hit and FA rates)')
print(f'Cell sizes: {sdt_df.n_intact.iloc[0]} intact, {sdt_df.n_rearranged.iloc[0]} rearranged per emotion per subject')
print()
print(sdt_summary.to_string())
print()

dprime_wide = sdt_df.pivot(index='subject', columns='emotion', values='d_prime')
angry_d = dprime_wide['angry'].tolist()
happy_d = dprime_wide['happy'].tolist()
neutral_d = dprime_wide['neutral'].tolist()

f_val_d, df1_d, df2_d, p_d, eta_d, mse_d = rm_anova_oneway([angry_d, happy_d, neutral_d])
print(f"One-way RM ANOVA on d' (flanker emotion):")
print(f"  F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, partial eta^2 = {eta_d:.3f}")
print()

print("Follow-up comparisons on d' (using omnibus MSE):")
d_groups = [angry_d, happy_d, neutral_d]
d_labels = ['angry', 'happy', 'neutral']
for i in range(3):
    for j in range(i + 1, 3):
        t, dfe, p, md = anova_followup(d_groups[i], d_groups[j], mse_d, df2_d,
                                        d_labels[i], d_labels[j])
        print(f'  {d_labels[i]} vs {d_labels[j]}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

crit_wide = sdt_df.pivot(index='subject', columns='emotion', values='criterion')
angry_c = crit_wide['angry'].tolist()
happy_c = crit_wide['happy'].tolist()
neutral_c = crit_wide['neutral'].tolist()

f_val_c, df1_c, df2_c, p_c, eta_c, mse_c = rm_anova_oneway([angry_c, happy_c, neutral_c])
print(f"One-way RM ANOVA on criterion (flanker emotion):")
print(f"  F({df1_c},{df2_c}) = {f_val_c:.3f}, p = {p_c:.3f}, partial eta^2 = {eta_c:.3f}")

SDT Analysis (n=6 subjects, per-emotion hit and FA rates)
Cell sizes: 10 intact, 10 rearranged per emotion per subject

         N  hit_rate_M  hit_rate_SD  fa_rate_M  fa_rate_SD  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                                                     
angry    6       0.517        0.204      0.533       0.225     -0.076       0.476       -0.091         0.563
happy    6       0.483        0.183      0.517       0.147     -0.084       0.491        0.000         0.358
neutral  6       0.517        0.117      0.467       0.175      0.134       0.426        0.028         0.335

One-way RM ANOVA on d' (flanker emotion):
  F(2,10) = 0.455, p = 0.647, partial eta^2 = 0.083

Follow-up comparisons on d' (using omnibus MSE):
  angry vs happy: t(10) = 0.031, p = 0.976, diff = 0.008
  angry vs neutral: t(10) = -0.810, p = 0.437, diff = -0.211
  happy vs neutral: t(10) = -0.842, p = 0.420, diff = -0.219

One-way RM

### SDT interpretation

d' is indistinguishable from zero for all three emotion conditions: angry d' = -0.08, happy d' = -0.08, neutral d' = 0.13. None differ from zero in any meaningful way, and the one-way ANOVA on d' shows no emotion effect, F(2,10) = 0.46, p = .647. Criterion is near zero for all emotions (no response bias), F(2,10) = 0.28, p = .765.

Hit rates (.48-.52) and FA rates (.47-.53) are all near .50, consistent with pure guessing. The balanced 10/10 cell sizes provide adequate sensitivity to detect even modest discrimination, yet there is none.

**Mar 11 comparison.** d' values are virtually unchanged: Mar 11 angry = -0.03, happy = -0.07, neutral = -0.11; Mar 13 angry = -0.08, happy = -0.08, neutral = 0.13. Both datasets converge on the same conclusion: subjects cannot distinguish intact from rearranged pairs in this task.

## Summary

### Practical implications

Doubling the study exposure from 60 to 120 trials (two complete passes through all 60 pairs) produced no improvement in associative discrimination. Combined with the Mar 11 pilot (which used a single study pass and the 24/36 split bug), this is now 12 condition 2 subjects across two pilots showing d' at zero.

**The problem is not insufficient encoding strength.** Subjects saw each pair twice and the study-phase accuracy and RT patterns confirm they were engaged in the orienting task. The failure is specific to associative memory -- subjects form no retrievable association between the target and flanker identities.

**Possible explanations:**
1. The incidental encoding task (gender judgment) does not encourage binding of the two face identities. Subjects focus on the target and treat the flanker as a distractor, even when it is incompatible.
2. The face pairs may be too similar across intact/rearranged conditions. Since rearrangement preserves trial type (same emotion, same gender combination), the recombined pairs may not feel different enough to discriminate.
3. Associative recognition for unfamiliar faces may require more explicit encoding instructions (e.g., "remember which faces appeared together") rather than incidental encoding.

**Decision point.** The current condition 2 design does not produce measurable associative memory. Options include: (a) modifying the encoding task to promote binding, (b) adding explicit study instructions for condition 2, (c) increasing the number of study passes further, or (d) reconsidering whether associative recognition is viable with this stimulus set and paradigm.

In [8]:
n_total_c2 = 6
n_excluded = n_total_c2 - df.subject_number.nunique()
n_kept = df.subject_number.nunique()

print(f'=== Condition 2 Pilot Summary (2026-03-13) ===')
print(f'7 subjects collected (6 c2 + 1 c1), analyzing {n_total_c2} c2 subjects')
print(f'{n_excluded} excluded (>=6 zero cells), {n_kept} analyzed')
print(f'Design: double study pass (120 trials), balanced 30/30 test split')
print()

print('Study phase (orienting):')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
r = acc_results['A']
print(f"  Compatibility: F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['B']
print(f"  Emotion:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['AxB']
print(f"  Interaction:   F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
print()

print('Test phase (associative recognition):')
r = psame_results['A']
print(f"  p(\"same\") Pair type:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['B']
print(f"  p(\"same\") Emotion:          F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['AxB']
print(f"  p(\"same\") Interaction:      F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['A']
print(f"  RT Pair type:              F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['B']
print(f"  RT Emotion:                F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['AxB']
print(f"  RT Interaction:            F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
print()

print('Supplementary SDT:')
for emotion, row in sdt_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}, fa={row['fa_rate_M']:.2f}")
print(f"  d' ANOVA: F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, eta^2 = {eta_d:.3f}")
print()

print('Comparison to Mar 11 pilot (single study, 24/36 bug):')
print(f"  Mar 11 d': angry=-0.03, happy=-0.07, neutral=-0.11")
print(f"  Mar 13 d': angry={sdt_summary.loc['angry','d_prime_M']:.2f}, "
      f"happy={sdt_summary.loc['happy','d_prime_M']:.2f}, "
      f"neutral={sdt_summary.loc['neutral','d_prime_M']:.2f}")

=== Condition 2 Pilot Summary (2026-03-13) ===
7 subjects collected (6 c2 + 1 c1), analyzing 6 c2 subjects
0 excluded (>=6 zero cells), 6 analyzed
Design: double study pass (120 trials), balanced 30/30 test split

Study phase (orienting):
  Overall accuracy: 75.8%
  Mean RT: 1050 ms
  Compatibility: F(1,5) = 4.730, p = 0.082
  Emotion:       F(2,10) = 0.354, p = 0.711
  Interaction:   F(2,10) = 0.723, p = 0.509

Test phase (associative recognition):
  p("same") Pair type:       F(1,5) = 0.000, p = 1.000, eta^2 = 0.000
  p("same") Emotion:          F(2,10) = 0.175, p = 0.842, eta^2 = 0.034
  p("same") Interaction:      F(2,10) = 0.432, p = 0.661, eta^2 = 0.080
  RT Pair type:              F(1,5) = 0.108, p = 0.755, eta^2 = 0.021
  RT Emotion:                F(2,10) = 0.732, p = 0.505, eta^2 = 0.128
  RT Interaction:            F(2,10) = 0.024, p = 0.976, eta^2 = 0.005

Supplementary SDT:
    angry: d'=-0.08 (SD=0.48), c=-0.09, hit=0.52, fa=0.53
    happy: d'=-0.08 (SD=0.49), c=0.00, hit